In [12]:
!pip install "pandas<3.0.0"

In [3]:
!pip install -q -U transformers peft trl bitsandbytes datasets scikit-learn

In [4]:
import os
import json
import pandas as pd
from kaggle_secrets import UserSecretsClient
from datasets import Dataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Authenticate with Hugging Face
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# Load and Split Data
iris = load_iris()
df = pd.DataFrame(iris.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
df['species'] = [iris.target_names[i] for i in iris.target]

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
test_df.to_csv("iris_test_set.csv", index=False)

# Format Functions for Gemma 4 Chat Template
def format_v1(row):
    prompt = f"sepal_length: {row['sepal_length']}, sepal_width: {row['sepal_width']}, petal_length: {row['petal_length']}, petal_width: {row['petal_width']}"
    answer = row['species']
    
    # Format for v1
    json_record = {"input_text": prompt, "output_text": answer}
    chat_text = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn>"
    return json_record, {"text": chat_text}

def format_v2(row):
    prompt = f"A flower specimen has a sepal length of {row['sepal_length']} cm, sepal width of {row['sepal_width']} cm, petal length of {row['petal_length']} cm, and petal width of {row['petal_width']} cm. Identify the iris species."
    answer = f"This is Iris {row['species']}."
    
    # Format for v2
    json_record = {"input_text": prompt, "output_text": answer}
    chat_text = f"<start_of_turn>system\nYou are a botanical classifier.<end_of_turn>\n<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn>"
    return json_record, {"text": chat_text}

# Generate JSONL files and Training Datasets
v1_records, v2_records, v1_train_data, v2_train_data = [], [], [], []

for _, row in train_df.iterrows():
    r1, t1 = format_v1(row)
    r2, t2 = format_v2(row)
    v1_records.append(r1); v1_train_data.append(t1)
    v2_records.append(r2); v2_train_data.append(t2)

with open("iris_v1_train.jsonl", "w") as f:
    for rec in v1_records: f.write(json.dumps(rec) + "\n")

with open("iris_v2_train.jsonl", "w") as f:
    for rec in v2_records: f.write(json.dumps(rec) + "\n")

dataset_v1 = Dataset.from_list(v1_train_data)
dataset_v2 = Dataset.from_list(v2_train_data)

print("✅ Task 1 & 2 Complete: Generated JSONL files and formatted training data.")

✅ Task 1 & 2 Complete: Generated JSONL files and formatted training data.


In [5]:
import os
# Fragmentation Fix
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
# Hide the second GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "google/gemma-4-e2b"

# Quantization Config 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True  
)

# LoRA Config 
peft_config = LoraConfig(
    r=4,                    
    lora_alpha=8,           
    lora_dropout=0.05,
    target_modules="all-linear", 
    task_type="CAUSAL_LM"
)

def train_and_save(dataset, version_name):
    print(f"\n🚀 Starting Training for {version_name.upper()}...")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.padding_side = "right"
    
    # Load the model with the dtype override
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        quantization_config=bnb_config, 
        device_map={'': 0},
        torch_dtype=torch.float16
    )
    
    # SFTConfig 
    training_args = SFTConfig(
        output_dir=f"./results-{version_name}",
        per_device_train_batch_size=1,         
        gradient_accumulation_steps=8,         
        gradient_checkpointing=True,           
        optim="paged_adamw_8bit",              
        learning_rate=2e-4,
        num_train_epochs=20,
        fp16=True,
        logging_steps=10,
        report_to="none",
        dataset_text_field="text",
        max_length=64               
    )
    
    # Initialize Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=peft_config,
        args=training_args,
    )
    
    print("Applying PyTorch precision fixes...")
    for name, param in trainer.model.named_parameters():
        if param.requires_grad:
            # Trainable adapters MUST be float32 for the GradScaler
            param.data = param.data.to(torch.float32)
        elif param.dtype == torch.bfloat16:
            # Frozen layers MUST be float16 for the Kaggle T4 GPU
            param.data = param.data.to(torch.float16)
    print("Scrub complete! Launching training...")
    
    # Launch Training
    trainer.train()
    
    # Save the physical artifacts
    output_dir = f"./gemma4-iris-{version_name}-adapter"
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✅ Saved {version_name.upper()} artifacts to {output_dir}")
    
    # Aggressively clear GPU memory for the next run
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

# # Run the training loops
# train_and_save(dataset_v1, "v1")

In [6]:
# train_and_save(dataset_v2, "v2")

In [ ]:
# import shutil

# source_path = '/kaggle/working/gemma4-iris-v2-adapter'
# output_filename = 'v2'
# shutil.make_archive(output_filename, 'zip', source_path)

In [10]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm 

MODEL_ID = "google/gemma-4-e2b"
valid_species = ['setosa', 'versicolor', 'virginica']

print("Loading base model for evaluation...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the base model once
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb_config, 
    device_map={'': 0},
    torch_dtype=torch.float16
)

def evaluate_adapter(adapter_path, version):
    print(f"\n📊 Evaluating {version.upper()}...")
    
    # Snap the LoRA adapter onto the base model
    model = PeftModel.from_pretrained(base_model, adapter_path)
    
    y_true, y_pred = [], []
    compliant_count = 0
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Testing {version.upper()}"):
        
        # Prepare Prompt based on version
        if version == "v1":
            few_shot_example = "<start_of_turn>user\nsepal_length: 5.1, sepal_width: 3.5, petal_length: 1.4, petal_width: 0.2<end_of_turn>\n<start_of_turn>model\nsetosa<end_of_turn>\n"
            prompt = f"sepal_length: {row['sepal_length']}, sepal_width: {row['sepal_width']}, petal_length: {row['petal_length']}, petal_width: {row['petal_width']}"
            
            # Prepend the example before the actual question
            input_text = few_shot_example + f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"

        
        inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
        
        # Generate Prediction 
        if version == "v1":
            outputs = model.generate(
                                **inputs, 
                                max_new_tokens=2, 
                                temperature=0.1, 
                                repetition_penalty=1.2,
                                eos_token_id=tokenizer.eos_token_id
                            )
            
        response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        # Robust Parsing
        response_clean = response.split("<start_of_turn>model\n")[-1]
        
        # Remove any remaining chat tags or thought blocks
        response_clean = re.sub(r'<\|channel>thought.*?<channel\|>', '', response_clean, flags=re.DOTALL)
        response_clean = response_clean.replace("<end_of_turn>", "").replace("<bos>", "").strip()
        
        # This ignores the "<start" and numeric rambling that follows
        first_word = response_clean.split()[0] if response_clean.split() else ""
        
        # Final cleaning (lowercase and remove trailing punctuation/tags)
        final_answer_cleaned = first_word.lower().split('<')[0].replace(".", "").strip()
        
        # Strict Grading
        y_true.append(row['species'])
        is_compliant, extracted_species = False, "unknown"
        
        if version == "v1":
            if final_answer_cleaned in valid_species:
                is_compliant, extracted_species = True, final_answer_cleaned
                    
        if is_compliant: 
            compliant_count += 1
        y_pred.append(extracted_species)

    # Calculate and Print Final Metrics
    compliance_rate = (compliant_count / len(test_df)) * 100
    acc = accuracy_score(y_true, y_pred)
    prec, rec, _, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=valid_species, zero_division=0)
    
    print(f"\n--- Results for {version.upper()} ---")
    print(f"Format Compliance: {compliance_rate:.2f}%")
    print(f"Overall Accuracy:  {acc * 100:.2f}%")
    for i, sp in enumerate(valid_species):
        print(f"  {sp:10} - Precision: {prec[i]:.2f}, Recall: {rec[i]:.2f}")
        
    # Free up memory so the next adapter can load smoothly
    del model
    torch.cuda.empty_cache()

# Run evaluation sequentially
evaluate_adapter("./gemma4-iris-v1-adapter", "v1")

Loading base model for evaluation...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]


📊 Evaluating V1...


Testing V1:   0%|          | 0/30 [00:00<?, ?it/s]


--- Results for V1 ---
Format Compliance: 63.33%
Overall Accuracy:  13.33%
  setosa     - Precision: 1.00, Recall: 0.10
  versicolor - Precision: 0.17, Recall: 0.33
  virginica  - Precision: 0.00, Recall: 0.00


In [11]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

MODEL_ID = "google/gemma-4-e2b"
valid_species = ['setosa', 'versicolor', 'virginica']

print("Loading fresh base model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load base model cleanly
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb_config, 
    device_map={'': 0},
    torch_dtype=torch.float16
)

def evaluate_v2_final(adapter_path):
    print(f"\n🚀 Evaluating V2")
    model = PeftModel.from_pretrained(base_model, adapter_path)
    
    y_true, y_pred = [], []
    compliant_count = 0
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        prompt = f"A flower specimen has a sepal length of {row['sepal_length']} cm, sepal width of {row['sepal_width']} cm, petal length of {row['petal_length']} cm, and petal width of {row['petal_width']} cm. Identify the iris species."
        
        # The V2 Few-Shot Injection
        input_text = (
            "<start_of_turn>system\nYou are a botanical classifier.<end_of_turn>\n"
            "<start_of_turn>user\nA flower specimen has a sepal length of 5.1 cm, sepal width of 3.5 cm, petal length of 1.4 cm, and petal width of 0.2 cm. Identify the iris species.<end_of_turn>\n"
            "<start_of_turn>model\nthis is iris setosa<end_of_turn>\n"
            f"<start_of_turn>user\n{prompt}<end_of_turn>\n"
            "<start_of_turn>model\n"
        )
        
        inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
        prompt_length = inputs['input_ids'].shape[1]
        
        # Generate with no repetition penalty
        outputs = model.generate(
            **inputs, 
            max_new_tokens=40, 
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )
        
        # Get the newly generated tokens
        new_tokens = outputs[0][prompt_length:]
        new_text = tokenizer.decode(new_tokens, skip_special_tokens=True).lower()
        
        # Stop reading if it hallucinates a new 'user' turn
        clean_answer = new_text.split("user")[0].strip()
        
        # Check if any valid species name exists in the clean text
        found_species = "unknown"
        for sp in valid_species:
            if sp in clean_answer:
                found_species = sp
                break
        
        y_true.append(row['species'])
        y_pred.append(found_species)
        
        if found_species != "unknown":
            compliant_count += 1

    # Calculate final metrics
    acc = accuracy_score(y_true, y_pred)
    compliance = (compliant_count / len(test_df)) * 100
    prec, rec, _, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=valid_species, zero_division=0)
    
    print(f"\n--- Final Results for V2 ---")
    print(f"Format Compliance: {compliance:.2f}%")
    print(f"Overall Accuracy:  {acc * 100:.2f}%")
    for i, sp in enumerate(valid_species):
        print(f"  {sp:10} - Precision: {prec[i]:.2f}, Recall: {rec[i]:.2f}")

    del model
    torch.cuda.empty_cache()

# Execute the final evaluation
evaluate_v2_final("./gemma4-iris-v2-adapter")

Loading fresh base model...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]


🚀 Evaluating V2


  0%|          | 0/30 [00:00<?, ?it/s]


--- Final Results for V2 ---
Format Compliance: 100.00%
Overall Accuracy:  63.33%
  setosa     - Precision: 0.71, Recall: 1.00
  versicolor - Precision: 0.38, Recall: 0.33
  virginica  - Precision: 0.75, Recall: 0.55
